# 01 — Data preparation

Phase 1 walkthrough. Set the three paths below to your local RSNA Stage 2 files. The dataset is not included in the repository. The paper specifies mean/std normalization at each pixel location; compute those statistics from the training split only and reuse them for all partitions.

In [ ]:
from pathlib import Path
from src.dataset import load_rsna_records, split_records, load_image, visualize_record
from src.preprocessing import compute_normalization_stats, save_normalization_stats

LABELS_CSV = Path('data/raw/stage_2_train_labels.csv')
CLASS_INFO_CSV = Path('data/raw/stage_2_detailed_class_info.csv')
IMAGE_DIR = Path('data/raw/stage_2_train_images')

In [ ]:
records, excluded_count = load_rsna_records(LABELS_CSV, CLASS_INFO_CSV, IMAGE_DIR)
train_records, validation_records, test_records = split_records(records, seed=42)
print(f'Included: {len(records)}; excluded other disease: {excluded_count}')
print(f'Train: {len(train_records)}; validation: {len(validation_records)}; test: {len(test_records)}')

# Reads each training DICOM once; may take time on the full RSNA dataset.
normalization_stats = compute_normalization_stats(record.image_path for record in train_records)
save_normalization_stats(normalization_stats, 'data/splits/normalization.npz')

In [ ]:
# Inspect the first record and its standardized 128 x 128 image.
record = records[0]
image = load_image(record, normalization_stats)
print(record.patient_id, record.class_name, image.shape, float(image.min()), float(image.max()))
fig, ax = visualize_record(record, normalization_stats)